### Setup
Autoreload and imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Imports
import sys, os
from pathlib import Path
from art.attacks.evasion import FastGradientMethod, ProjectedGradientDescent, SaliencyMapMethod, CarliniL2Method
from art.defences.trainer import AdversarialTrainer, AdversarialTrainerTRADESPyTorch
import art.attacks.evasion.projected_gradient_descent.projected_gradient_descent_pytorch as _pgd_pt
_pgd_pt.compute_success = lambda *a, **kw: 0.0

sys.path.append(str(Path.cwd().parents[1]))

from utils.functions import get_windowed_data
from utils.notebook import get_model_classifier, clean_data_test, adv_test, FilenameLoader, get_filename_from_path, freeze_attack_cols

/opt/anaconda3/envs/reu/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Define Inputs and Load Data

In [3]:
## Load checkpoint and data file paths
_, data_name, _ = FilenameLoader.rand_pos()

checkpoint_file= f"../../saved_models/adv_trained/RandPos-PGD-30-30-200.ckpt"
data_file = f"../../data/{data_name}"

In [ ]:
## Load data
(x_train, y_train), (x_test, y_test), fed_dataset, scaler = get_windowed_data(data_file, 
                                                                      normalize=True, 
                                                                      train_perc=80)

### Adversarial training

In [14]:
print("> Before Adv Test")

model, classifier = get_model_classifier(checkpoint_file)
adv_save_dir = f"../fed/data-test/randpos_trained-30-30-200_no-retraining"
name = get_filename_from_path(checkpoint_file)

after_adv_f1 = []
for i in range(1, 31):
    eps = float(i/100)
    after_adv_out = adv_test(
        classifier, x_test, y_test, 
        checkpoint_file=checkpoint_file, data_file=data_file,
        end_index=len(y_test.numpy()),
        path=adv_save_dir,
        filename=f"after_adv_eps_{eps}_advtrained_{name}.json",
        Attack=FastGradientMethod,
        eps=eps,
    )
    after_adv_f1.append(after_adv_out["metrics"]["f1"])

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


> Before Adv Test
Checkpoint path exists!
=== Attack: FastGradientMethod, kwargs: {'eps': 0.01} ===
Accuracy:           0.9401
Precision:          0.9997
Recall:             0.7986
F1:                 0.8879
ASR (FNR):          0.2014
False Positive Rate:0.0001
TP=296032, TN=876958, FP=82, FN=74668
Time elapsed:       15.47s
Saved metrics to ../fed/data-test/randpos_trained-30-30-200_no-retraining/after_adv_eps_0.01_advtrained_RandPos-PGD-30-30-200.json
=== Attack: FastGradientMethod, kwargs: {'eps': 0.02} ===
Accuracy:           0.9342
Precision:          0.9997
Recall:             0.7789
F1:                 0.8756
ASR (FNR):          0.2211
False Positive Rate:0.0001
TP=288727, TN=876943, FP=97, FN=81973
Time elapsed:       14.82s
Saved metrics to ../fed/data-test/randpos_trained-30-30-200_no-retraining/after_adv_eps_0.02_advtrained_RandPos-PGD-30-30-200.json
=== Attack: FastGradientMethod, kwargs: {'eps': 0.03} ===
Accuracy:           0.9291
Precision:          0.9996
Recall:       

In [15]:
# Get clean baseline 
print("> Before Clean Test")
before_clean_out = clean_data_test(
    model, classifier, x_test, y_test, 
    checkpoint_file=checkpoint_file, data_file=data_file,
    save_path=adv_save_dir,
    filename=f"before_clean_advtrained_{name}.json",
    save_results=True
)

> Before Clean Test
torch.Size([124774, 10, 2])
0.9997890274495067
0.8181629349878609
Model got 1180269/1247740 right.
Accuracy: 0.9459254331831952, Precision: 0.9997890274495067, Recall: 0.8181629349878609, F1 Score: 0.8999031239197873
877040, 70.29028483498165% Zeroes, 370700 Non Zero entries.
Saved to ../fed/data-test/randpos_trained-30-30-200_no-retraining/before_clean_advtrained_RandPos-PGD-30-30-200.json


In [ ]:
## Simple training - pgd at 0.05
from art.defences.trainer import AdversarialTrainer
import torch 

# Training vars
adv_train_eps = 0.05
adv_train_epochs = 5
adv_train_ratio = 0.5 

x_train_np = x_train.numpy()
y_train_np = y_train.numpy()

# Init vars
name = get_filename_from_path(checkpoint_file)
model, classifier = get_model_classifier(checkpoint_file)

print(f"=== Adversarial training: {name} | eps:  {adv_train_eps} ===")

adv_save_dir = f"../fed/data-test/randpos_pgd-0.05_epochs-5_trained-30-30-200"
os.makedirs(adv_save_dir, exist_ok=True)

# Get clean baseline 
print("> Before Clean Test")
before_clean_out = clean_data_test(
    model, classifier, x_test, y_test, 
    checkpoint_file=checkpoint_file, data_file=data_file,
    save_path=adv_save_dir,
    filename=f"before_clean_advtrained_{name}.json",
    save_results=True
)

# Check condition 1
print ("> Before Adv Test")
before_adv_out = adv_test(
    classifier, x_test, y_test, 
    checkpoint_file=checkpoint_file, data_file=data_file,
    end_index=len(y_test.numpy()),
    path=adv_save_dir,
    filename=f"before_adv_eps_{adv_train_eps}_advtrained_{name}.json",
    Attack=ProjectedGradientDescent,
    eps=adv_train_eps,
    max_iter = 5
)

# Run adv training
# freeze_attack_cols keeps rcvTime (col 0) untouched, same threat model as
# adv_test's freeze_cols default - a real attacker can't manipulate the
# receive timestamp, so FGSM shouldn't be allowed to "cheat" through it
# during training either.
print("> Running Adv Training")
attack = freeze_attack_cols(ProjectedGradientDescent(classifier, eps=adv_train_eps, max_iter=5), freeze_cols=(0,))
trainer = AdversarialTrainer(classifier, attacks=attack, ratio=adv_train_ratio)
trainer.fit(x_train_np, y_train_np, nb_epochs=adv_train_epochs)

# Re-evaluate after adversarial training
print("> After Clean Test")
after_clean_out = clean_data_test(
    model, classifier, x_test, y_test, 
    checkpoint_file=checkpoint_file, data_file=data_file,
    save_path=adv_save_dir,
    filename=f"after_clean_advtrained_{name}.json",
    save_results=True
)

print("> After Adv Test")
after_adv_f1 = []
for i in range(1, 31):
    eps = float(i/100)
    after_adv_out = adv_test(
        classifier, x_test, y_test, 
        checkpoint_file=checkpoint_file, data_file=data_file,
        end_index=len(y_test.numpy()),
        path=adv_save_dir,
        filename=f"after_adv_eps_{eps}_advtrained_{name}.json",
        Attack=FastGradientMethod,
        eps=eps,
    )
    after_adv_f1.append(after_adv_out["metrics"]["f1"])


# Save the adversarially-trained weights (loadable via utils.functions.load_model_checkpoint)
ckpt_out = f"{adv_save_dir}/advtrained_{name}.ckpt"
torch.save(model.learner.state_dict(), ckpt_out)
print(f"Saved adversarially-trained checkpoint to {ckpt_out}")



GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Checkpoint path exists!
=== Adversarial training: RandPos-PGD-30-30-200 | eps:  0.05 ===
> Before Clean Test
torch.Size([124774, 10, 2])
0.9997890274495067
0.8181629349878609
Model got 1180269/1247740 right.
Accuracy: 0.9459254331831952, Precision: 0.9997890274495067, Recall: 0.8181629349878609, F1 Score: 0.8999031239197873
877040, 70.29028483498165% Zeroes, 370700 Non Zero entries.
Saved to ../fed/data-test/randpos_pgd-0.05_epochs-30_trained-30-30-200/before_clean_advtrained_RandPos-PGD-30-30-200.json
> Before Adv Test
=== Attack: ProjectedGradientDescent, kwargs: {'eps': 0.05, 'max_iter': 5} ===


Accuracy:           0.9202
Precision:          0.9992
Recall:             0.7320
F1:                 0.8450
ASR (FNR):          0.2680
False Positive Rate:0.0002
TP=271358, TN=876836, FP=204, FN=99342
Time elapsed:       75.22s
Saved metrics to ../fed/data-test/randpos_pgd-0.05_epochs-30_trained-30-30-200/before_adv_eps_0.05_advtrained_RandPos-PGD-30-30-200.json
> Running Adv Training


Adversarial training epochs:   7%|▋         | 2/30 [05:41<1:19:40, 170.71s/it]

In [10]:
import json
config = {
    "eps": adv_train_eps,
    "adv_train_epochs": adv_train_epochs,
    "adv_train_ratio": adv_train_ratio,
    "checkpoint_file": checkpoint_file,
    "attack": "ProjectedGradientDescent",
    "name": name,
}

with open(f"{adv_save_dir}/config.json", "w") as f:
    json.dump(config, f, indent=4)
    print("saved:", f"{adv_save_dir}/config.json")

saved: ../fed/data-test/randpos-pgd-0.05/config.json


In [ ]:
## Find lowerbound threshold - lowest eps that still
lo_eps = 0.07 # fgsm, from the previously done eps sweep, f1 = 0.7079755805153897 < 0.77


In [ ]:
## Find upperbound threshold - highest eps that starts making benign wrong

import json, time, torch
# When f1 is at least 20 pts down from og (so 0.77)

# ratio = 0.5, epochs = 5, eps = 0.2 -> ran until eps = 0.99, ratio to low? (high-eps-iterative)
# ratio = 1.0, epochs = 5, eps = 0.1 -> clean f1 score dec immediately, ratio to high? (high-eps-iterative-ratio-1.0)
# ratio = 0.7, epochs = 5, eps = 0.1 -> running current (high-eps-iterative-ratio-0.7)

# Defined
clean_f1 = 0.976470669379591
threshold_diff = 0.2
working_eps = 0.9

# Training Hyperparams
adv_train_epochs = 5
adv_train_ratio = 0.7

# Var definitions
x_train_np = x_train.numpy()
y_train_np = y_train.numpy()
name = get_filename_from_path(checkpoint_file)
index = 1

# run once

while index < 2:
    start = time.time()
    print(f"\n\n>>> New eps: {working_eps}")
    # Reload classifier
    index = index + 1
    model, classifier = get_model_classifier(checkpoint_file)
    adv_save_dir = f"../fed/data-test/freeze-col-test/iter-{index}"
    os.makedirs(adv_save_dir, exist_ok=True)
    print(f"Saving to: {adv_save_dir}")

    ## Adv train model
    print("> Running Adv Training")
    attack = freeze_attack_cols(FastGradientMethod(classifier, eps=working_eps), freeze_cols=(0,))
    trainer = AdversarialTrainer(classifier, attack, ratio=adv_train_ratio)
    trainer.fit(x_train_np, y_train_np, nb_epochs=adv_train_epochs, batch_size=64)

    # Save model
    ckpt_out = f"{adv_save_dir}/advtrained_{name}.ckpt"
    torch.save(model.learner.state_dict(), ckpt_out)
    print(f"Saved adversarially-trained checkpoint to {ckpt_out}")

    ## Test model
    after_clean_out = clean_data_test(
        model, classifier, x_test, y_test, 
        checkpoint_file=checkpoint_file, data_file=data_file,
        save_path=adv_save_dir,
        filename=f"after_clean_advtrained_{name}.json",
        save_results=True
    )

    # Save config
    

    # Check if f1 is below
    adv_f1 = after_clean_out["wrapper"]["f1"]
    if (adv_f1 < clean_f1 - threshold_diff): 
        print(f"BREAKING WITH EPS {working_eps}")
        break
    else:
        print(f"f1 = {adv_f1} >  {clean_f1 - threshold_diff}")
        working_eps = round(working_eps + 0.01, 2)
        print(f"Increasing eps to {working_eps}")

print("goodbye world")


    
    

"""
- Start at ~ 0.2 eps
- Adv train model on the # of eps (save the model)
- Run clean test (and SAVE it)
- Check if it's below the threshold

* Can try just doing 10 points.. later (or just see from data)
"""

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.




>>> New eps: 0.9
Checkpoint path exists!
Saving to: ../fed/data-test/freeze-col-test/iter-2
> Running Adv Training


Adversarial training epochs: 100%|██████████| 5/5 [07:05<00:00, 85.16s/it]


Saved adversarially-trained checkpoint to ../fed/data-test/freeze-col-test/iter-2/advtrained_RandomPos-final.ckpt
Saved to ../fed/data-test/freeze-col-test/iter-2/after_clean_advtrained_RandomPos-final.json
f1 = 0.9955288715845922 >  0.7764706693795911
Increasing eps to 0.91
goodbye world


"\n- Start at ~ 0.2 eps\n- Adv train model on the # of eps (save the model)\n- Run clean test (and SAVE it)\n- Check if it's below the threshold\n\n* Can try just doing 10 points.. later (or just see from data)\n"

In [ ]:
# y_train_np.sum(axis=1)[102]

np.int64(0)

In [7]:
## Find upperbound threshold - highest eps that starts making benign wrong

import json, time, torch
import numpy as np
# When f1 is at least 20 pts down from og (so 0.77)

# ratio = 0.5, epochs = 5, eps = 0.2 -> ran until eps = 0.99, ratio to low? (high-eps-iterative)
# ratio = 1.0, epochs = 5, eps = 0.1 -> clean f1 score dec immediately, ratio to high? (high-eps-iterative-ratio-1.0)
# ratio = 0.7, epochs = 5, eps = 0.1 -> running current (high-eps-iterative-ratio-0.7)

# Defined
clean_f1 = 0.976470669379591
threshold_diff = 0.2
working_eps = 0.1

# Training Hyperparams
adv_train_epochs = 5

# Var definitions
x_train_np = x_train.numpy()
y_train_np = (y_train.numpy() == 1).all(axis=1).astype(np.int64)
name = get_filename_from_path(checkpoint_file)
index = 2

# run once

while index < 3:
    start = time.time()
    print(f"\n\n>>> New eps: {working_eps}")
    # Reload classifier
    index = index + 1
    model, classifier = get_model_classifier(checkpoint_file, collapsed=True)
    classifier._reduce_labels = True
    adv_save_dir = f"../fed/data-test/trades-test/iter-{index}"
    os.makedirs(adv_save_dir, exist_ok=True)
    print(f"Saving to: {adv_save_dir}")

    ## Adv train model
    print("> Running Adv Training")
    attack = FastGradientMethod(classifier, eps=working_eps)
    trainer = AdversarialTrainerTRADESPyTorch(classifier, attack=attack, beta=0.6)
    trainer.fit(x_train_np, y_train_np, nb_epochs=adv_train_epochs, batch_size=64)

    # Save model
    ckpt_out = f"{adv_save_dir}/advtrained_{name}.ckpt"
    torch.save(model.learner.state_dict(), ckpt_out)
    print(f"Saved adversarially-trained checkpoint to {ckpt_out}")

    ## Test model
    after_clean_out = clean_data_test(
        model, classifier, x_test, y_test, 
        checkpoint_file=checkpoint_file, data_file=data_file,
        save_path=adv_save_dir,
        filename=f"after_clean_advtrained_{name}.json",
        save_results=True,
        collapsed=True
    )

    # Save config
    config = {
        "eps": working_eps,
        "adv_train_epochs": adv_train_epochs,
        "beta": 0.6,
        "clean_f1_baseline": clean_f1,
        "threshold_diff": threshold_diff,
        "index": index,
        "checkpoint_file": checkpoint_file,
        "name": name,
        "timeElapsedSec": (time.time() - start)
    }

    with open(f"{adv_save_dir}/config.json", "w") as f:
        json.dump(config, f, indent=4)

    # Check if f1 is below
    adv_f1 = after_clean_out["wrapper"]["f1"]
    if (adv_f1 < clean_f1 - threshold_diff): 
        print(f"BREAKING WITH EPS {working_eps}")
        break
    else:
        print(f"f1 = {adv_f1} >  {clean_f1 - threshold_diff}")
        working_eps = round(working_eps + 0.01, 2)
        print(f"Increasing eps to {working_eps}")

print("goodbye world")


    
    

"""
- Start at ~ 0.2 eps
- Adv train model on the # of eps (save the model)
- Run clean test (and SAVE it)
- Check if it's below the threshold

* Can try just doing 10 points.. later (or just see from data)
"""

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.




>>> New eps: 0.1
Checkpoint path exists!
Saving to: ../fed/data-test/trades-test/iter-3
> Running Adv Training


Adversarial Training TRADES - Epochs: 100%|██████████| 5/5 [12:13<00:00, 146.73s/it]


Saved adversarially-trained checkpoint to ../fed/data-test/trades-test/iter-3/advtrained_RandomPos-final.ckpt
Saved to ../fed/data-test/trades-test/iter-3/after_clean_advtrained_RandomPos-final.json
f1 = 0.9970922762743267 >  0.7764706693795911
Increasing eps to 0.11
goodbye world


"\n- Start at ~ 0.2 eps\n- Adv train model on the # of eps (save the model)\n- Run clean test (and SAVE it)\n- Check if it's below the threshold\n\n* Can try just doing 10 points.. later (or just see from data)\n"

In [16]:
min(3, 1)

1